# Lab 2 — Trader project (market-data MCP)

This lab is the trader project.

- MCP server file: `market_data_server.py`
- Free data path: `get_stooq_quote` (no key)
- Optional key path: `get_polygon_previous_close` with `POLYGON_API_KEY`

Goal: run a trader-style agent that chooses tools to answer market prompts.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

EXERCISE_DIR = Path.cwd().resolve()

REPO_ROOT = EXERCISE_DIR.parent.parent.parent

market_mcp_params = {
    "command": "uv",
    "args": ["run", str(EXERCISE_DIR / "market_data_server.py")],
    "cwd": str(REPO_ROOT),
}

### Run trader prompt with free Stooq data

In [ ]:
async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as server:
    market_tools = await server.list_tools()

market_tools

In [ ]:
from IPython.display import Markdown, display

request = "Use get_stooq_quote for aapl.us and summarize the latest daily quote for a cautious trader in 3 bullet points."

async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    trader_agent = Agent(
        name="trader_assistant",
        instructions=(
            "You are a careful market assistant. Use MCP tools for prices, "
            "state if data is delayed, and avoid financial-advice certainty."
        ),
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
    )
    with trace("adeyemi_lab2_trader"):
        result = await Runner.run(trader_agent, request)

display(Markdown(result.final_output))

In [ ]:
# Optional: test Polygon path if POLYGON_API_KEY is set
request_polygon = "Use get_polygon_previous_close for AAPL and explain what this metric means in one sentence."

async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    trader_agent = Agent(
        name="trader_assistant_polygon",
        instructions="Use market MCP tools and be explicit when a key is missing.",
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
    )
    result_polygon = await Runner.run(trader_agent, request_polygon)

display(Markdown(result_polygon.final_output))

### Notes

- Use Stooq symbol format like `aapl.us`.
- Polygon call needs `POLYGON_API_KEY`.